# Track 2 · Stage 3 — Fine-tune Whisper (M6 / M7 / M8)

Pulls everything from the Hub. Nothing is generated here.

| model | training data | paper MER |
|---|---|---|
| M6 | Train_T1 (8h synthetic)        | 48.2 |
| M7 | Train_T2 (22h synthetic)       | 40.8 |
| M8 | Train_T2 + Common Voice mono   | 39.2 |

`whisper-small` (244M) is **fully fine-tuned** — no LoRA, no quantization. Optimizer
state is ~4 GB, which fits a 16 GB T4, so the paper's recipe survives intact:
AdamW, lr 2e-5, effective batch 64.

**T4 is Turing**: no bf16, no FlashAttention-2. `fp16=True` gives autocast + GradScaler.

Deviation D2: the paper used whisper-large-v2 (1.54B). Absolute MER will be far worse;
what we test is the **M6 → M7 → M8 ordering**.

**Temporary further deviation, budget-driven, not a pipeline decision:** `MODEL` is
currently `openai/whisper-base` (74M) and steps/eval/patience are capped well below the
paper's 5000-step ceiling. Kaggle's free-tier GPU quota turned out to be a **~6h/day**
allowance rather than a 30h/week pool, which doesn't fit whisper-small's full recipe
before a hard deadline. Swap `MODEL` back to `openai/whisper-small` and `MAX_STEPS` back
to `"5000"` (with `EVAL_STEPS="250"`, `PATIENCE="4"`) once that stops being the binding
constraint -- nothing else in this notebook or in `train_whisper.py` needs to change to
do that.

### Two things that used to break this notebook

**Featurization used to hang forever.** `datasets.map(num_proc=2)` forks after the
parent has already used the Rust fast tokenizer, and the child inherits its thread
pool and deadlocks — the bar sits at `0/N` with **zero CPU**, which looks like "slow"
but never finishes. Fixed in `csasr>=0.10.3`: `TOKENIZERS_PARALLELISM=false` is set at
import time and `--num-proc` now defaults to **1**. Featurization is numpy log-mel plus
a FLAC decode, so one worker costs only a few minutes across the whole corpus.

**The featurized cache fills the disk.** One whisper-small log-mel is
80 × 3000 float32 = **0.96 MB per clip** — ~5.8 GB for M6, ~17.3 GB for M7, ~35.5 GB
for M8. Left to accumulate that is ~61 GB and M8 dies with *No space left on device*
partway through. `free_map_cache()` runs before M7 and M8 and deletes only the
`cache-*.arrow` files, keeping the expensive parquet download.

**A killed session used to mean starting the current model over from step 0.**
Kaggle can kill a session (walltime cap, disconnect) with no warning and no chance
to run cleanup code, and `/kaggle/working` dies with it. Every `run(...)` call below
now passes `--hub-checkpoint-repo`: the Trainer's full checkpoint (model + optimizer
+ scheduler + RNG + `trainer_state.json`, not just weights) is pushed to a private
Hub repo every `--eval-steps` and cleared once that model finishes cleanly.
Re-running this notebook after a crash therefore does the right thing automatically:
`model_already_trained()` skips whichever of M6/M7/M8 already finished, and the one
that was interrupted resumes from its last pushed checkpoint instead of restarting.

In [ ]:
# datasets>=4 decodes Audio via torchcodec, which stock Kaggle images lack.
!pip install -q -U transformers accelerate evaluate jiwer
!pip install -q "datasets<4" librosa soundfile soxr omegaconf rich

# csasr LAST and FORCED: `pip install git+...` treats an already-installed
# version as satisfied and SKIPS the reinstall, so re-running in a live kernel
# silently keeps OLD code. --no-deps because the deps are installed above.
!pip install -q --force-reinstall --no-deps git+https://github.com/BRUH-MAIN/codeswitching.git

# This NOTEBOOK's own version. `pip install` updates the csasr PACKAGE but NOT the
# .ipynb -- an old notebook against a new package is a real and confusing failure.
NOTEBOOK_VERSION = "0.11.0"

import csasr
assert csasr.__version__ == NOTEBOOK_VERSION, (
    f"csasr package is {csasr.__version__} but this NOTEBOOK is {NOTEBOOK_VERSION}. "
    "If the PACKAGE is older: restart the kernel (Run > Restart & clear) -- pip skips "
    "a reinstall when the version already looks satisfied. If the NOTEBOOK is older: "
    "re-download it from the repo -- pip does NOT update .ipynb files."
)
print("csasr", csasr.__version__)

import os, subprocess, sys
from pathlib import Path
os.environ["HF_HOME"] = "/kaggle/temp/hf"

from kaggle_secrets import UserSecretsClient
# Token goes in the environment, never in argv (it leaks into every traceback).
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import HfApi, hf_hub_download, login
login(token=os.environ["HF_TOKEN"])
HF_TOKEN = os.environ["HF_TOKEN"]   # for load_dataset / hf_hub_download only
hub_api = HfApi(token=HF_TOKEN)

SYNTH = "RohanRamesh/hi-en-synth-cs"
REAL  = "RohanRamesh/mucs-he-cs"
# DEVIATION (temporary, budget-driven): whisper-base (74M), not whisper-small
# (244M) -- Kaggle's free-tier GPU quota turned out to be a ~6h/day allowance,
# not a 30h/week pool, which doesn't fit whisper-small's full recipe before a
# hard deadline. Swap back to "openai/whisper-small" once there's more budget.
MODEL = "openai/whisper-base"
# Resumable checkpoints for ALL of smoke/M6/M7/M8 live here, one subfolder per
# run (m6/last-checkpoint, m6/best-checkpoint, m7/...). See hub_checkpoint.py.
CKPT_REPO = "RohanRamesh/csasr-train-checkpoints"
# Same budget reason: capped well below the paper's 5000-step ceiling instead
# of trusting early stopping's unknown timing to cut it short on its own. M8
# stayed behind this flag (rather than being removed) until M6/M7 confirmed the
# capped recipe actually works -- they did (both finished cleanly, M7 < M6 as
# expected), so M8 is on. Its per-step cost is the same as M6/M7's (the 800-step
# cap bounds wall-clock, not dataset size); only featurization is bigger.
MAX_STEPS, EVAL_STEPS, PATIENCE = "800", "200", "3"
TRAIN_M8 = True

def run(*args):
    print(">", " ".join(str(a) for a in args), flush=True)
    subprocess.run([sys.executable, "-m", *args], check=True)

def model_already_trained(repo_id: str) -> bool:
    """Has `repo_id` already been fully trained and pushed?

    Lets a re-run of this notebook after a killed session skip whichever
    model(s) already finished -- only the interrupted one needs to resume.
    """
    try:
        files = hub_api.list_repo_files(repo_id, token=HF_TOKEN)
    except Exception:
        return False
    return any(f.endswith((".safetensors", ".bin")) for f in files)

def free_map_cache():
    """Delete the featurized Arrow cache, keeping the downloaded parquet.

    A whisper-small log-mel is 80 x 3000 float32 = 0.96 MB PER CLIP, so the
    featurized cache is ~5.8 GB for M6, ~17.3 GB for M7 and ~35.5 GB for M8.
    Left to accumulate that is ~61 GB and M8 dies with No space left on device
    after an hour of work. The parquet download is the expensive part, so only
    the cache-*.arrow files are removed.
    """
    import glob
    freed = 0
    for f in glob.glob("/kaggle/temp/hf/**/cache-*.arrow", recursive=True):
        try:
            freed += os.path.getsize(f); os.remove(f)
        except OSError:
            pass
    print(f"[cache] freed {freed / 1e9:.1f} GB of featurized Arrow")
    !df -h /kaggle/temp | tail -1

t1_ids = hf_hub_download(SYNTH, "t1_ids.json", repo_type="dataset", token=HF_TOKEN)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/temp | tail -1

## Smoke test first

20 steps on 1% of the data proves the collator, the `<|hi|>` prefix tokens, and the label masking work — before committing a 3h session.

In [ ]:
run("csasr.train.train_whisper", "--model", MODEL, "--out", "/kaggle/working/smoke",
    "--train-hf", SYNTH, "--train-config", "synth_t2",
    "--dev-hf", REAL, "--dev-config", "dev",
    "--max-steps", "20", "--eval-steps", "10", "--dataset-fraction", "0.01",
    "--batch-size", "4", "--grad-accum", "1",
    "--hub-checkpoint-repo", CKPT_REPO)

## M6 — Train_T1 (8h synthetic)

Skipped automatically if `RohanRamesh/whisper-base-cs-m6` already exists (e.g. this notebook is re-running after a killed session).

In [ ]:
if model_already_trained("RohanRamesh/whisper-base-cs-m6"):
    print("[skip] RohanRamesh/whisper-base-cs-m6 already exists")
else:
    run("csasr.train.train_whisper", "--model", MODEL, "--out", "/kaggle/working/m6",
        "--train-hf", SYNTH, "--train-config", "synth_t2", "--subset-ids", t1_ids,
        "--dev-hf", REAL, "--dev-config", "dev",
        "--lr", "2e-5", "--batch-size", "16", "--grad-accum", "4",
        "--max-steps", MAX_STEPS, "--eval-steps", EVAL_STEPS, "--patience", PATIENCE,
        "--hub-checkpoint-repo", CKPT_REPO)

## M7 — Train_T2 (22h synthetic)

Clear M6's featurized cache first — see `free_map_cache()` above for why. Skipped automatically if `RohanRamesh/whisper-base-cs-m7` already exists.

In [ ]:
free_map_cache()
if model_already_trained("RohanRamesh/whisper-base-cs-m7"):
    print("[skip] RohanRamesh/whisper-base-cs-m7 already exists")
else:
    run("csasr.train.train_whisper", "--model", MODEL, "--out", "/kaggle/working/m7",
        "--train-hf", SYNTH, "--train-config", "synth_t2",
        "--dev-hf", REAL, "--dev-config", "dev",
        "--lr", "2e-5", "--batch-size", "16", "--grad-accum", "4",
        "--max-steps", MAX_STEPS, "--eval-steps", EVAL_STEPS, "--patience", PATIENCE,
        "--hub-checkpoint-repo", CKPT_REPO)

## M8 — Train_T2 + Common Voice monolingual (52h)

Still one `<|hi|>` prompt for everything: language-*specific* prompting is Track 1's M4.

**This is the run that fills the disk.** ~37,000 clips x 0.96 MB = **~35.5 GB** of
featurized Arrow, so clearing M7's cache first is not optional.

Gated behind `TRAIN_M8` (currently `True`) -- the most expensive of the three models
(~6x M6's per-epoch cost). M6 and M7 both finished cleanly under the capped recipe with
the expected M7 < M6 ordering, so M8 is now on too. Its per-step cost matches M6/M7's
(the 800-step cap bounds wall-clock, not dataset size); the extra time is mostly a bigger
one-time featurization pass. Skipped automatically if `RohanRamesh/whisper-base-cs-m8`
already exists, same as M6/M7 skip if already trained.

In [ ]:
if not TRAIN_M8:
    print("[skip] TRAIN_M8 is False")
else:
    free_map_cache()
    if model_already_trained("RohanRamesh/whisper-base-cs-m8"):
        print("[skip] RohanRamesh/whisper-base-cs-m8 already exists")
    else:
        run("csasr.train.train_whisper", "--model", MODEL, "--out", "/kaggle/working/m8",
            "--train-hf", SYNTH, "--train-config", "synth_t2",
            "--extra-hf", f"{REAL}:cv_hi", f"{REAL}:cv_en",
            "--dev-hf", REAL, "--dev-config", "dev",
            "--lr", "2e-5", "--batch-size", "16", "--grad-accum", "4",
            "--max-steps", MAX_STEPS, "--eval-steps", EVAL_STEPS, "--patience", PATIENCE,
            "--hub-checkpoint-repo", CKPT_REPO)

## Push checkpoints so `03_eval` can find them

A model that was skipped above (already trained in an earlier session, or parked behind
`TRAIN_M8`) either doesn't exist locally or is already on the Hub, so this loop is a
harmless no-op for it -- `upload_folder` just re-uploads identical files where there's
anything to push at all.

In [ ]:
for m in ("m6", "m7", "m8"):
    local = Path(f"/kaggle/working/{m}")
    if not local.exists():
        print(f"[skip push] {m}: no local output (not trained this session)")
        continue
    repo = f"RohanRamesh/whisper-base-cs-{m}"
    hub_api.create_repo(repo, exist_ok=True, private=True, token=HF_TOKEN)
    hub_api.upload_folder(folder_path=str(local), repo_id=repo, token=HF_TOKEN)
    print("pushed", repo)